In [ ]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# 1. Load the data
df = pd.read_csv("sms_spam_clean__2_ (6).csv")
print("Shape:", df.shape)
print(df.head())

# 2. Check for missing values / duplicates
print("Missing values:\n", df.isnull().sum())
df = df.dropna(subset=["message"])
df = df.drop_duplicates(subset=["message"])

# 3. Simple text cleaning function
def clean_text(text):
    text = text.lower()                                  # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)            # remove URLs
    text = re.sub(r"\d+", "", text)                       # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()               # remove extra spaces
    return text

df["clean_message"] = df["message"].apply(clean_text)

# 4. Quick look at cleaned data
print(df[["message", "clean_message"]].head())

# 5. Split into features (X) and target (y)
X = df["clean_message"]
y = df["label_num"]   # 0 = ham, 1 = spam

# 6. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Convert text to numeric features (Bag of Words)
vectorizer = CountVectorizer(stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training data shape:", X_train_vec.shape)
print("Testing data shape:", X_test_vec.shape)

# 8. Save cleaned dataset (optional)
df.to_csv("sms_spam_cleaned_output.csv", index=False)
print("Preprocessing done! Cleaned file saved as sms_spam_cleaned_output.csv")

Shape: (5176, 3)
                                             message  label_num  msg_length
0  Go until jurong point, crazy.. Available only ...          0         111
1                      Ok lar... Joking wif u oni...          0          29
2  Free entry in 2 a wkly comp to win FA Cup fina...          1         155
3  U dun say so early hor... U c already then say...          0          49
4  Nah I don't think he goes to usf, he lives aro...          0          61
Missing values:
 message       0
label_num     0
msg_length    0
dtype: int64
                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go until jurong point crazy available only in ...  
1 

In [ ]:
import pandas as pd
import re
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# 1. Load the data
df = pd.read_csv("sms_spam_cleaned_output.csv")
print("Shape:", df.shape)

 # remove filler words (stopwords) like "the", "is", "a", "to"...
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return " ".join(words)

df["clean_message"] = df["message"].apply(clean_text)

# 4. Quick before/after look
print(df[["message", "clean_message"]].head(10))

# 5. Save cleaned dataset
df.to_csv("sms_spam_no_fillers.csv", index=False)
print("\nDone! Filler words removed. Saved as sms_spam_no_fillers.csv")

Shape: (5176, 4)
                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   
5  FreeMsg Hey there darling it's been 3 week's n...   
6  Even my brother is not like to speak with me. ...   
7  As per your request 'Melle Melle (Oru Minnamin...   
8  WINNER!! As a valued network customer you have...   
9  Had your mobile 11 months or more? U R entitle...   

                                       clean_message  
0  jurong point crazy available bugis n great wor...  
1                            ok lar joking wif u oni  
2  free entry wkly comp win fa cup final tkts st ...  
3                        u dun say early hor u c say  
4                      nah dont think goes usf lives  
5  freemsg hey darling weeks word id

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt

# 1. Load the filler-free cleaned data
df = pd.read_csv("sms_spam_no_fillers.csv")
df["clean_message"] = df["clean_message"].fillna("")  # handle empty cleaned messages

# =========================================================
# PART 1: EDA (Exploratory Data Analysis)
# =========================================================

print("Shape of data:", df.shape)

print("\nClass counts (0=ham, 1=spam):")
print(df["label_num"].value_counts())

print("\nClass percentage:")
print(df["label_num"].value_counts(normalize=True) * 100)

# length of the cleaned (filler-removed) message
df["clean_length"] = df["clean_message"].apply(lambda x: len(str(x)))

print("\nAverage clean_message length by class:")
print(df.groupby("label_num")["clean_length"].mean())

# word count in cleaned message
df["word_count"] = df["clean_message"].apply(lambda x: len(str(x).split()))

print("\nAverage word count (after filler removal) by class:")
print(df.groupby("label_num")["word_count"].mean())

# Plot 1: Class balance
plt.figure()
df["label_num"].value_counts().plot(kind="bar")
plt.title("Ham (0) vs Spam (1) Count")
plt.xlabel("Label")
plt.ylabel("Number of Messages")
plt.savefig("class_balance.png")
plt.close()

# Plot 2: Cleaned message word count distribution by class
plt.figure()
df[df["label_num"] == 0]["word_count"].hist(alpha=0.5, label="Ham", bins=30)
df[df["label_num"] == 1]["word_count"].hist(alpha=0.5, label="Spam", bins=30)
plt.legend()
plt.title("Word Count Distribution (after filler removal)")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.savefig("word_count_distribution.png")
plt.close()

# Most common words in spam vs ham (now meaningful, since fillers are gone)
def top_words(text_series, n=15):
    all_words = " ".join(text_series.astype(str)).split()
    return pd.Series(all_words).value_counts().head(n)

print("\nTop 15 words in SPAM messages (filler-free):")
print(top_words(df[df["label_num"] == 1]["clean_message"]))

print("\nTop 15 words in HAM messages (filler-free):")
print(top_words(df[df["label_num"] == 0]["clean_message"]))

# =========================================================
# PART 2: FEATURE ENGINEERING
# =========================================================

# Feature 1: number of digits in original message
df["num_digits"] = df["message"].apply(lambda x: sum(c.isdigit() for c in str(x)))

# Feature 2: number of exclamation marks in original message
df["num_exclamations"] = df["message"].apply(lambda x: str(x).count("!"))

# Feature 3: has a URL or not (1/0)
df["has_url"] = df["message"].apply(lambda x: 1 if re.search(r"http\S+|www\S+", str(x).lower()) else 0)

# Feature 4: percentage of uppercase letters in original message
def pct_upper(text):
    text = str(text)
    letters = [c for c in text if c.isalpha()]
    if len(letters) == 0:
        return 0
    upper = [c for c in letters if c.isupper()]
    return len(upper) / len(letters)

df["pct_upper"] = df["message"].apply(pct_upper)

# Feature 5: contains common spam trigger words (checked on clean_message)
trigger_words = ["free", "win", "winner", "urgent", "claim", "prize", "txt", "cash", "call"]
df["has_trigger_word"] = df["clean_message"].apply(
    lambda x: 1 if any(word in str(x).split() for word in trigger_words) else 0
)

# Feature 6: word_count and clean_length are already added above as features

print("\nNew feature columns added:")
cols_to_show = ["clean_message", "word_count", "clean_length", "num_digits",
                 "num_exclamations", "has_url", "pct_upper", "has_trigger_word"]
print(df[cols_to_show].head())

# Compare new features by class
print("\nAverage feature values by class:")
print(df.groupby("label_num")[["word_count", "clean_length", "num_digits",
                                "num_exclamations", "has_url", "pct_upper",
                                "has_trigger_word"]].mean())

# 3. Save the dataset with new features
df.to_csv("sms_spam_final_features.csv", index=False)
print("\nDone! Saved as sms_spam_final_features.csv")
print("Plots saved as class_balance.png and word_count_distribution.png")

Shape of data: (5176, 4)

Class counts (0=ham, 1=spam):
label_num
0    4518
1     658
Name: count, dtype: int64

Class percentage:
label_num
0    87.287481
1    12.712519
Name: proportion, dtype: float64

Average clean_message length by class:
label_num
0    40.253209
1    81.050152
Name: clean_length, dtype: float64

Average word count (after filler removal) by class:
label_num
0     7.242807
1    13.316109
Name: word_count, dtype: float64

Top 15 words in SPAM messages (filler-free):
£         217
free      187
u         128
txt       124
ur        119
stop      108
text      104
mobile    102
claim      97
reply      93
prize      79
just       65
new        64
p          61
send       57
Name: count, dtype: int64

Top 15 words in HAM messages (filler-free):
u       903
im      442
just    275
dont    253
ok      247
got     227
know    221
like    221
come    211
good    207
ill     207
ur      202
time    187
day     173
love    173
Name: count, dtype: int64

New feature columns a

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 1. Load the cleaned data
df = pd.read_csv("sms_spam_final_features.csv")
df["clean_message"] = df["clean_message"].fillna("")

# 2. Features and target
X = df["clean_message"]
y = df["label_num"]

# 3. Train/test split (original, untouched imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Vectorize text
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 5. Train Logistic Regression, but tell it to automatically weight
#    the minority class (spam) higher so it doesn't get ignored
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_vec, y_train)

# 6. Predict
y_pred = model.predict(X_test_vec)

# 7. Evaluate
print("===== Logistic Regression with class_weight='balanced' =====")
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1-score: ", f1_score(y_test, y_pred))

print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))

print("\nFull Report:")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

# 8. Try it on sample messages
sample_messages = [
    "Congratulations! You won a free prize, claim now by texting WIN to 12345",
    "Hey, are we still meeting for lunch tomorrow?"
]
sample_vec = vectorizer.transform(sample_messages)
predictions = model.predict(sample_vec)

print("\nSample predictions:")
for msg, pred in zip(sample_messages, predictions):
    label = "SPAM" if pred == 1 else "HAM"
    print(f"[{label}] {msg}")

===== Logistic Regression with class_weight='balanced' =====
Accuracy:  0.9797297297297297
Precision: 0.9586776859504132
Recall:    0.8787878787878788
F1-score:  0.9169960474308301

Confusion Matrix (rows=actual, cols=predicted):
[[899   5]
 [ 16 116]]

Full Report:
              precision    recall  f1-score   support

         Ham       0.98      0.99      0.99       904
        Spam       0.96      0.88      0.92       132

    accuracy                           0.98      1036
   macro avg       0.97      0.94      0.95      1036
weighted avg       0.98      0.98      0.98      1036


Sample predictions:
[SPAM] Congratulations! You won a free prize, claim now by texting WIN to 12345
[HAM] Hey, are we still meeting for lunch tomorrow?


In [ ]:
import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 1. Load the cleaned data
df = pd.read_csv("sms_spam_final_features.csv")
df["clean_message"] = df["clean_message"].fillna("")

# 2. Split into train/test FIRST (important: only balance the training data, never the test data)
X = df["clean_message"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Put train data back together to oversample
train_df = pd.DataFrame({"clean_message": X_train, "label_num": y_train})

ham_train = train_df[train_df["label_num"] == 0]
spam_train = train_df[train_df["label_num"] == 1]

print("Before oversampling:")
print("Ham:", len(ham_train), " Spam:", len(spam_train))

# 4. Oversample the spam (minority) class to match ham count
spam_oversampled = resample(
    spam_train,
    replace=True,              # sample with replacement
    n_samples=len(ham_train),  # match majority class count
    random_state=42
)

train_balanced = pd.concat([ham_train, spam_oversampled])
train_balanced = train_balanced.sample(frac=1, random_state=42)  # shuffle rows

print("\nAfter oversampling:")
print(train_balanced["label_num"].value_counts())

X_train_bal = train_balanced["clean_message"]
y_train_bal = train_balanced["label_num"]

# 5. Vectorize text (fit only on training data)
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train_bal)
X_test_vec = vectorizer.transform(X_test)

# 6. Train Naive Bayes on the BALANCED data
model = MultinomialNB()
model.fit(X_train_vec, y_train_bal)

# 7. Predict on the untouched, original test set
y_pred = model.predict(X_test_vec)

# 8. Evaluate
print("\n===== Results after fixing class imbalance =====")
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1-score: ", f1_score(y_test, y_pred))

print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))

print("\nFull Report:")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

Before oversampling:
Ham: 3614  Spam: 526

After oversampling:
label_num
0    3614
1    3614
Name: count, dtype: int64

===== Results after fixing class imbalance =====
Accuracy:  0.9555984555984556
Precision: 0.7621951219512195
Recall:    0.946969696969697
F1-score:  0.8445945945945946

Confusion Matrix (rows=actual, cols=predicted):
[[865  39]
 [  7 125]]

Full Report:
              precision    recall  f1-score   support

         Ham       0.99      0.96      0.97       904
        Spam       0.76      0.95      0.84       132

    accuracy                           0.96      1036
   macro avg       0.88      0.95      0.91      1036
weighted avg       0.96      0.96      0.96      1036



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

# 1. Load data
df = pd.read_csv("sms_spam_final_features.csv")
df["clean_message"] = df["clean_message"].fillna("")

X = df["clean_message"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_df = pd.DataFrame({"clean_message": X_train, "label_num": y_train})
ham_train = train_df[train_df["label_num"] == 0]
spam_train = train_df[train_df["label_num"] == 1]

# Oversample spam to match ham
spam_oversampled = resample(
    spam_train, replace=True, n_samples=len(ham_train), random_state=42
)
train_balanced = pd.concat([ham_train, spam_oversampled])

# =========================================================
# GRAPH 1: Class counts BEFORE vs AFTER balancing (training set)
# =========================================================
before_counts = [len(ham_train), len(spam_train)]
after_counts = [len(ham_train), len(spam_oversampled)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].bar(["Ham", "Spam"], before_counts, color=["#4C72B0", "#DD8452"])
axes[0].set_title("Before Balancing (Training Set)")
axes[0].set_ylabel("Number of Messages")
for i, v in enumerate(before_counts):
    axes[0].text(i, v + 30, str(v), ha="center")

axes[1].bar(["Ham", "Spam"], after_counts, color=["#4C72B0", "#DD8452"])
axes[1].set_title("After Oversampling (Training Set)")
axes[1].set_ylabel("Number of Messages")
for i, v in enumerate(after_counts):
    axes[1].text(i, v + 30, str(v), ha="center")

plt.tight_layout()
plt.savefig("class_balance_before_after.png")
plt.close()

# =========================================================
# GRAPH 2: Model comparison (Precision, Recall, F1 for spam class)
# Using the results already computed in earlier scripts
# =========================================================
models = ["Naive Bayes\n(baseline)", "Naive Bayes\n(oversampled)", "Logistic Regression\n(class_weight balanced)"]
precision = [0.944, 0.762, 0.959]
recall    = [0.902, 0.947, 0.879]
f1        = [0.922, 0.845, 0.917]

import numpy as np
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width, precision, width, label="Precision", color="#4C72B0")
ax.bar(x,          recall,    width, label="Recall",    color="#DD8452")
ax.bar(x + width,  f1,        width, label="F1-score",  color="#55A868")

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Spam Detection: Precision / Recall / F1 by Approach")
ax.legend()

for i, vals in enumerate([precision, recall, f1]):
    for j, v in enumerate(vals):
        offset = (i - 1) * width
        ax.text(j + offset, v + 0.02, f"{v:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("model_comparison_metrics.png")
plt.close()

print("Saved: class_balance_before_after.png")
print("Saved: model_comparison_metrics.png")

Saved: class_balance_before_after.png
Saved: model_comparison_metrics.png


In [ ]:
# ===============================================
# SMS Spam Classifier - Train & Save as .joblib
# Run this in Google Colab
# ===============================================

# 1. Install/import required libraries
import pandas as pd
import re
import string
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 2. Upload your CSV file (Colab file picker)
from google.colab import files
uploaded = files.upload()   # choose your sms_spam_clean.csv when prompted

# Get the filename that was uploaded
filename = list(uploaded.keys())[0]

# 3. Load the data
df = pd.read_csv(filename)
print("Shape:", df.shape)

# 4. Drop missing values and duplicates
df = df.dropna(subset=["message"])
df = df.drop_duplicates(subset=["message"])

# 5. Cleaning function (lowercase, remove urls/numbers/punctuation, remove filler words)
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return " ".join(words)

df["clean_message"] = df["message"].apply(clean_text)

# 6. Features and target
X = df["clean_message"]
y = df["label_num"]   # 0 = ham, 1 = spam

# 7. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 8. Vectorize text (Bag of Words)
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 9. Train the model (Logistic Regression with class_weight='balanced')
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_vec, y_train)

# 10. Evaluate
y_pred = model.predict(X_test_vec)
print("\n===== Model Performance =====")
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1-score: ", f1_score(y_test, y_pred))
print("\n", classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

# 11. Save BOTH the model and the vectorizer as .joblib files
#     (you need both to make predictions on new messages later)
joblib.dump(model, "sms_spam_model.joblib")
joblib.dump(vectorizer, "sms_vectorizer.joblib")
print("\nSaved: sms_spam_model.joblib")
print("Saved: sms_vectorizer.joblib")

# 12. Download the files to your computer
files.download("sms_spam_model.joblib")
files.download("sms_vectorizer.joblib")

# ===============================================
# HOW TO LOAD AND USE THEM LATER (in a new script/session)
# ===============================================
# model = joblib.load("sms_spam_model.joblib")
# vectorizer = joblib.load("sms_vectorizer.joblib")
#
# new_msg = ["Congratulations! You won a free prize, claim now"]
# new_msg_clean = [clean_text(m) for m in new_msg]
# new_vec = vectorizer.transform(new_msg_clean)
# prediction = model.predict(new_vec)
# print("SPAM" if prediction[0] == 1 else "HAM")

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from google.colab import files

# Combine your already-trained vectorizer and model into ONE pipeline object
# (swap in your actual variable names if they're different)
pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("classifier", model)
])

# Save the whole pipeline as a single .joblib file
joblib.dump(pipeline, "sms_spam_pipeline.joblib")
print("Saved: sms_spam_pipeline.joblib")

# Download it to your computer
files.download("sms_spam_pipeline.joblib")

# ===============================================
# HOW TO USE IT LATER (just one file needed now!)
# ===============================================
# pipeline = joblib.load("sms_spam_pipeline.joblib")
#
# new_msg = ["Congratulations! You won a free prize, claim now"]
# prediction = pipeline.predict(new_msg)   # raw text goes straight in
# print("SPAM" if prediction[0] == 1 else "HAM")

Saved: sms_spam_pipeline.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>